### Connexion à la DB DuckDB

In [1]:
import duckdb
import os
from pathlib import Path
from typing import List
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from tqdm import tqdm
import sklearn
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

### Connexion à la DB / Import des Data


In [2]:
# Store database at project root
DB_NAME = Path("/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/amazing.duckdb") 
# Go up one level from current directory to get to project root
data_folder = Path("..") / "data"
# For absolute certainty, you could use the absolute path
# data_folder = Path("/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/data")
con = duckdb.connect(str(DB_NAME))

In [3]:
# 2. Query to list all tables in the database
# DuckDB specific way to list tables
tables_info = con.sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").df()

print(f"Found {len(tables_info)} tables in the database:\n")

if len(tables_info) > 0:
    for i, table_name in enumerate(tables_info['table_name']):
        print(f"{i+1}. {table_name}")
else:
    print("No tables found in the database.")

Found 3 tables in the database:

1. all_events
2. loaded_files
3. user_segments_kmeans


In [4]:
# 5. Alternative way to show all tables
print("List of all tables using DuckDB's connections.tables():")
con.sql("SHOW TABLES").show()

List of all tables using DuckDB's connections.tables():
┌──────────────────────┐
│         name         │
│       varchar        │
├──────────────────────┤
│ all_events           │
│ loaded_files         │
│ user_segments_kmeans │
└──────────────────────┘



In [5]:
# Examine the all_events table
print("First 10 rows of all_events table:")
all_events_data = con.sql("""
    SELECT * FROM all_events LIMIT 10
""")
all_events_data.show()


# Examine the loaded_files table
print("\nContents of loaded_files table:")
loaded_files_data = con.sql("""
    SELECT * FROM loaded_files
""")
loaded_files_data.show()


First 10 rows of all_events table:
┌─────────────────────┬────────────┬────────────┬─────────────────────┬────────────────────────────────┬─────────┬─────────┬───────────┬──────────────────────────────────────┐
│     event_time      │ event_type │ product_id │     category_id     │         category_code          │  brand  │  price  │  user_id  │             user_session             │
│      timestamp      │  varchar   │  varchar   │       varchar       │            varchar             │ varchar │ double  │  varchar  │               varchar                │
├─────────────────────┼────────────┼────────────┼─────────────────────┼────────────────────────────────┼─────────┼─────────┼───────────┼──────────────────────────────────────┤
│ 2019-12-01 00:00:00 │ view       │ 1005105    │ 2232732093077520756 │ construction.tools.light       │ apple   │ 1302.48 │ 556695836 │ ca5eefc5-11f9-450c-91ed-380285a0bc80 │
│ 2019-12-01 00:00:00 │ view       │ 22700068   │ 2232732091643068746 │ NULL         

In [6]:
# Examine the all_events table
print("First 10 rows of all_events table:")
all_events_data = con.sql("""
    SELECT * FROM all_events LIMIT 10
""")
all_events_data.show()

# Show count of records in all_events
record_count = con.sql("""
    SELECT COUNT(*) as total_events FROM all_events
""")
record_count.show()

# Examine the loaded_files table
print("\nContents of loaded_files table:")
loaded_files_data = con.sql("""
    SELECT * FROM loaded_files
""")
loaded_files_data.show()


First 10 rows of all_events table:
┌─────────────────────┬────────────┬────────────┬─────────────────────┬────────────────────────────────┬─────────┬─────────┬───────────┬──────────────────────────────────────┐
│     event_time      │ event_type │ product_id │     category_id     │         category_code          │  brand  │  price  │  user_id  │             user_session             │
│      timestamp      │  varchar   │  varchar   │       varchar       │            varchar             │ varchar │ double  │  varchar  │               varchar                │
├─────────────────────┼────────────┼────────────┼─────────────────────┼────────────────────────────────┼─────────┼─────────┼───────────┼──────────────────────────────────────┤
│ 2019-12-01 00:00:00 │ view       │ 1005105    │ 2232732093077520756 │ construction.tools.light       │ apple   │ 1302.48 │ 556695836 │ ca5eefc5-11f9-450c-91ed-380285a0bc80 │
│ 2019-12-01 00:00:00 │ view       │ 22700068   │ 2232732091643068746 │ NULL         

In [7]:
DB_NAME = "amazing.duckdb"
TABLE_EVENTS = "all_events"
TABLE_USER_EVENTS = "user_events"
SAMPLE_USER_PERCENT = 1
BATCH_SIZE = 1000 

### Import de la table DuckDB

In [8]:
# Chargement de users avec au moins 10 événements 
print("Chargement d'un échantillon d'utilisateurs actifs...")

# Afficher les tables disponibles dans la base de données
tables_df = con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'").fetch_df()
print("Tables disponibles dans la base de données :")
print(tables_df)

user_ids_df = con.execute(f"""
    SELECT user_id
    FROM all_events
    WHERE user_id IS NOT NULL
    GROUP BY user_id
    HAVING COUNT(*) >= 10
""").fetch_df()

Chargement d'un échantillon d'utilisateurs actifs...
Tables disponibles dans la base de données :
             table_name
0            all_events
1          loaded_files
2  user_segments_kmeans


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### Normalisation et Standardisation des données

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

sampled_user_ids = user_ids_df.sample(frac=SAMPLE_USER_PERCENT, random_state=42)['user_id'].tolist()

print(f"Nombre d'utilisateurs actifs échantillonnés : {len(sampled_user_ids)}")

#  Création des features utilisateurs batch par batch 
print("Création des features utilisateurs par batch...")

user_features_list = []

for i in tqdm(range(0, len(sampled_user_ids), BATCH_SIZE), desc="Avancement user features", ncols=100):
    batch_ids = sampled_user_ids[i:i+BATCH_SIZE]
    batch_ids_str = ",".join(f"'{uid}'" for uid in batch_ids)

    batch_query = f"""
    WITH
        base_events AS (
            SELECT
                user_id,
                event_type,
                event_time,
                price,
                category_code,
                LEAD(event_time) OVER (PARTITION BY user_id ORDER BY event_time) AS next_event_time
            FROM {TABLE_EVENTS}
            WHERE user_id IN ({batch_ids_str})
        ),
        features AS (
            SELECT
                user_id,
                COUNT(*) AS total_events,
                SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS total_views,
                SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS total_purchases,
                AVG(EXTRACT(EPOCH FROM (next_event_time - event_time))) AS avg_time_between_events,
                SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END) AS total_spent,
                COALESCE(AVG(CASE WHEN event_type = 'purchase' THEN price ELSE NULL END), 0) AS avg_basket,
                MAX(event_time) AS last_event_time
            FROM base_events
            GROUP BY user_id
    )
    SELECT
        *,
        CASE WHEN total_views > 0 THEN total_purchases * 1.0 / total_views ELSE 0 END AS conversion_rate,
        CASE WHEN (total_views + total_purchases) > 0 THEN total_purchases * 1.0 / (total_views + total_purchases) ELSE 0 END AS purchase_ratio,
        DATE_PART('day', CAST('2020-03-31 22:00:00' AS TIMESTAMP) - last_event_time) AS days_since_last_event
    FROM features
    """

    batch_features = con.execute(batch_query).fetch_df()

    # Récupérer les user_id de cette batch
    valid_user_ids = con.execute(f"""
        SELECT user_id
        FROM {TABLE_EVENTS}
        WHERE user_id IN ({batch_ids_str})
        GROUP BY user_id
        HAVING COUNT(*) >= 10
    """).fetch_df()
    valid_user_ids = set(valid_user_ids["user_id"].astype(str))

    # Filtrage strict des user_id valides
    batch_features = batch_features[batch_features["user_id"].astype(str).isin(valid_user_ids)]

    user_features_list.append(batch_features)

# Fusionner tous les batchs
user_features = pd.concat(user_features_list, ignore_index=True)

# Vérification des NaN
print("Vérification des NaN")
nan_summary = user_features.isna().sum()
print("Résumé des NaN par colonne :")
print(nan_summary[nan_summary > 0])

users_with_nan = user_features[user_features.isna().any(axis=1)]
print(f"Nombre d'utilisateurs avec des NaN : {len(users_with_nan)}")
print("Exemples d'utilisateurs avec NaN :")
print(users_with_nan.head(10))

Nombre d'utilisateurs actifs échantillonnés : 4167451
Création des features utilisateurs par batch...


Avancement user features: 100%|███████████████████████████████| 4168/4168 [3:15:59<00:00,  2.82s/it]


Vérification des NaN
Résumé des NaN par colonne :
Series([], dtype: int64)
Nombre d'utilisateurs avec des NaN : 0
Exemples d'utilisateurs avec NaN :
Empty DataFrame
Columns: [user_id, total_events, total_views, total_purchases, avg_time_between_events, total_spent, avg_basket, last_event_time, conversion_rate, purchase_ratio, days_since_last_event]
Index: []


In [10]:
# Standardisation
print("Standardisation des features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(user_features.drop(columns=["last_event_time"]))

Standardisation des features...


In [11]:
user_features

,user_id,total_events,total_views,total_purchases,avg_time_between_events,total_spent,avg_basket,last_event_time,conversion_rate,purchase_ratio,days_since_last_event
0,575829553,36,35.0,0.0,230035.657143,0.00,0.00,2020-02-25 12:55:39,0.000000,0.000000,35
1,560327900,17,9.0,2.0,538.125000,263.28,131.64,2019-10-15 06:29:38,0.222222,0.181818,168
2,540202592,33,32.0,0.0,81734.843750,0.00,0.00,2019-11-07 12:04:08,0.000000,0.000000,145
3,581915786,21,14.0,0.0,146354.550000,0.00,0.00,2020-01-07 07:54:57,0.000000,0.000000,84
4,514097810,12,12.0,0.0,103.272727,0.00,0.00,2019-11-27 22:17:58,0.000000,0.000000,124
...,...,...,...,...,...,...,...,...,...,...,...
4167446,568940076,23,22.0,0.0,282857.181818,0.00,0.00,2020-02-11 12:15:24,0.000000,0.000000,49
4167447,603099642,10,7.0,1.0,68376.666667,253.55,253.55,2020-01-26 09:11:56,0.142857,0.125000,65
4167448,591979055,11,11.0,0.0,42.500000,0.00,0.00,2019-12-26 07:45:06,0.000000,0.000000,96
4167449,567515597,17,17.0,0.0,462367.312500,0.00,0.00,2020-01-31 22:15:17,0.000000,0.000000,59


In [12]:
#  Sauvegarde des résultats dans DuckDB
print(f"Sauvegarde dans {TABLE_USER_EVENTS}...")
con.execute(f"DROP TABLE IF EXISTS {TABLE_USER_EVENTS}")
con.register("temp_user_features", user_features)
con.execute(f"CREATE TABLE {TABLE_USER_EVENTS} AS SELECT * FROM temp_user_features")

Sauvegarde dans user_events...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [13]:
con.close()